In [1]:
using LowLevelFEM

## Central Difference Method

The Central Difference Method is an explicit second-order time integration
scheme widely used for transient structural dynamics.

It is conditionally stable and therefore requires the time step to be smaller
than the critical time step determined by the highest natural frequency of the
finite element model.

In [2]:
structured_rect_mesh(n=40)

mat = Material("body", E=2e5, ν=0.3, ρ=7.86e-9)

prob = Problem([mat], type=:PlaneStress);

## Geometry and material

A rectangular plate is discretized using a structured finite element mesh.

The material is assumed to be homogeneous, isotropic, and linearly elastic
under plane stress conditions.

In [3]:
K = stiffnessMatrix(prob)
M = massMatrix(prob, lumped=true);

## System matrices

The finite element stiffness matrix and the lumped mass matrix are assembled.

A lumped mass matrix is used because it is particularly suitable for explicit
time integration methods such as the Central Difference Method.

## Boundary conditions

The left edge is fully constrained to eliminate rigid body motion.

The right edge is constrained in the horizontal direction when computing the
critical time step.

In [4]:
bc1 = displacementConstraint("left", ux=0, uy=0)
bc2 = displacementConstraint("right", ux=0);

In [5]:
Tmin = smallestPeriodTime(K, M, support=[bc1])
dt = Tmin / π

4.721106507065234e-9

## Critical time step

The smallest natural period of the structure is estimated and used to compute
a stable explicit time step

$$
\Delta t = \frac{T_{\min}}{\pi}.
$$

This value satisfies the stability condition of the Central Difference Method.

In [6]:
n = 200;

## External loading

A triangular pulse is applied on the right boundary.

The load increases linearly during the first four time steps and then
decreases linearly to zero over the next four time steps.

In [7]:
sc = 10 / dt
f_fx(x, y, z, t) = t < 4*dt ? sc*t : t < 8*dt ? sc*(8*dt - t) : 0
fx_right = ScalarField(prob, "right", f_fx, tmax=n*dt, steps=n);

In [8]:
ld = load("right", fx=fx_right);

## Load vector

The time-dependent boundary traction is converted into the corresponding
finite element load vector.

In [9]:
f = loadVector(prob, [ld]);

## Initial conditions

The structure is initially at rest, with zero displacement and zero velocity.

In [10]:
u0 = initialDisplacement(prob, "body", ux=0, uy=0)
v0 = initialVelocity(prob, "body", vx=0, vy=0);

## Time integration

The dynamic response is computed using the explicit Central Difference Method.

The displacement and velocity fields are evaluated for the prescribed number
of time steps.

In [11]:
u, v = CDM(K, M, f, u0, v0, n, dt, support=[bc1]);

## Stress recovery

The transient stress field is evaluated from the computed displacement field
at every time step.

In [12]:
s = solveStress(u);

## Visualization

The transient displacement, velocity, acceleration, and stress fields are
displayed in the Gmsh post-processing window.

The displacement field is shown with an amplified deformation for better
visualization.

In [13]:
showDoFResults(f, name="f")
showDoFResults(u, name="u", factor=4000, visible=true)
showDoFResults(v, name="v")
showDoFResults(s, name="σ");

In [14]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
